# Round 9: combine the two leads, deliberate missing-data handling - S6E10

Round 8 left the campaign with two independent descriptive leads: seed averaging (Exp 22, +0.00031
vs incumbent, the current preferred candidate) and value-count features (Exp 27, +0.00059 vs
incumbent, highest measured OOF 0.95906, but within the band vs Exp 22 - a lead, not a win). Two
bounded tests, existing data only:

- **Exp 29: counts + stochastic 3-seed averaging.** The untested combination of the only two
  mechanisms that ever produced all-positive fold deltas in this campaign. If they stack, this is
  the strongest on-data candidate yet; if they overlap (both may be capturing generator structure),
  the combination adds little.
- **Exp 30: deliberate Arrival Delay missingness.** The standing queued experiment: 292 training
  rows (and their test counterparts) have missing Arrival Delay. LightGBM routes NaN natively; the
  one-change test adds an explicit missingness indicator column so the model can split on
  "was missing" directly.

Bar and reading rules unchanged: refit requires beating Exp 22's round-6 fold constants under the
paired 2x band (a decision heuristic, not an inferential test); cross-run pairing caveat as stated
in round 8. Exp 29 is additionally paired against Exp 27's round-8 folds, so the log can separate
"beats the bar" from "beats the stronger descriptive lead".


In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # current preferred candidate (bar)
E27_FOLDS_R8 = [0.95901, 0.95808, 0.95997, 0.95898, 0.95924]        # strongest descriptive lead (counts)
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)


In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
DELAY_COL = 'Arrival Delay in Minutes'
if DELAY_COL not in features.columns:  # column-name variants across file versions
    DELAY_COL = [c for c in features.columns if 'Arrival Delay' in c][0]

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False, **params):
    # count_features=True computes the 21 value-count columns fold-locally (train-only tables)
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        X_tr, X_va = to_lgbm(X_tr), to_lgbm(X_va)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; delay column: {DELAY_COL!r} '
      f'({int(features[DELAY_COL].isna().sum())} missing)')


## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every round.


In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc
INC_MEAN = float(np.mean(aucs_inc))
print(f'  incumbent mean OOF AUC: {INC_MEAN:.5f} +/- {np.std(aucs_inc):.5f} | pooled: {roc_auc_score(target, oof_inc):.6f}')


## Exp 29: value-count features + stochastic 3-seed averaging

The Exp 27 feature set inside the Exp 22 training scheme: fold-local counts, stochastic config,
seeds 42/43/44, OOF probabilities averaged per row. Paired against the incumbent, the Exp 22 bar,
and Exp 27's round-8 folds (the stronger descriptive lead).


In [ ]:
print('Exp 29: counts + stochastic, seeds 42/43/44, averaged')
oof_seeds = {}
iters29 = None
for seed in [42, 43, 44]:
    o, a, it = oof_lgbm(features, target, f'E29-s{seed}', seed=seed, count_features=True, **STOCH)
    oof_seeds[seed] = o
    if seed == 42:
        iters29 = it  # kept for the refit best-iteration basis
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof29 = np.mean([oof_seeds[s] for s in [42, 43, 44]], axis=0)
aucs29 = fold_aucs(oof29, target)
for fold, a in enumerate(aucs29, 1):
    print(f'  E29 fold {fold} (3-seed avg): AUC {a:.5f}')
print(f'Exp 29 mean OOF AUC: {np.mean(aucs29):.5f} +/- {np.std(aucs29):.5f} | pooled: {roc_auc_score(target, oof29):.6f}')


## Exp 30: deliberate Arrival Delay missingness

One change on the incumbent: an explicit missingness indicator for the delay column, NaN itself
left native. Hypothesis: "missing" may itself be informative (these 292 rows were altered after
synthesis, per the competition's own description pattern for this series); an indicator lets the
model split on it directly instead of inferring it from the NaN branch.


In [ ]:
print(f'Exp 30: incumbent config + {DELAY_COL} missingness indicator')
features_m = features.copy()
features_m['delay_was_missing'] = features_m[DELAY_COL].isna().astype(int)
oof30, aucs30, iters30 = oof_lgbm(features_m, target, 'E30')
print(f'Exp 30 mean OOF AUC: {np.mean(aucs30):.5f} +/- {np.std(aucs30):.5f} | pooled: {roc_auc_score(target, oof30):.6f}')


## Results, verdicts, and refit guard

Pairings: Exp 29 vs incumbent, vs the Exp 22 bar, and vs Exp 27's round-8 folds; Exp 30 vs
incumbent and vs the bar. Refit requires beating the Exp 22 bar under the paired band. The log
keeps the two-score distinction explicit: highest measured OOF vs the preferred refit candidate.


In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

results = {}
for exp_id, name, aucs, oof, refs in [
    ('29', 'counts + stochastic 3-seed avg', aucs29, oof29, [('Exp 27 (counts, round-8 folds)', E27_FOLDS_R8)]),
    ('30', 'incumbent + delay missingness flag', aucs30, oof30, []),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs, INC_FOLDS)
    d22, m22, s22, lab22 = paired_verdict(aucs, E22_FOLDS_R6)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label, m22=m22, lab22=lab22)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.6f}')
    print(f'  paired vs in-run incumbent: ' + ' '.join(f'{x:+.5f}' for x in deltas) + f' (mean {mean_d:+.5f}) | {label}')
    print(f'  paired vs Exp 22 (the bar): ' + ' '.join(f'{x:+.5f}' for x in d22) + f' (mean {m22:+.5f}) | {lab22}')
    for ref_name, ref_aucs in refs:
        d3, m3, s3, lab3 = paired_verdict(aucs, ref_aucs)
        print(f'  paired vs {ref_name}: mean {m3:+.5f} | {lab3}')
print()

best_oof = max([0.95906] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Highest measured OOF (incl. Exp 27 lead): {best_oof:.5f} - >=0.96: {best_oof >= 0.96} '
      f'(descriptive; preferred candidate stays Exp 22 at 0.95878 unless the bar is beaten)')

winners = [k for k, r in results.items()
           if r['m22'] > 0 and 'WORKING SIGNAL: better' in r['lab22']]
if not winners:
    print('No candidate beat the Exp 22 bar: Exp 22 (submission_round6.csv, local on Mac) remains the preferred submission candidate; no submission file written this round.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    print(f'Winner vs the Exp 22 bar: Exp {win} ({results[win]["name"]}) - refits on all rows below; logged as a working signal, exploratory.')


In [ ]:
# Refit ONLY fires on a whole-data working-signal winner over the Exp 22 bar.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win == '29':
        # 3 full-data stochastic models on full-train counts; n_estimators = mean of THIS run's
        # seed-42 fold best_iterations (same seed-shared basis semantics as rounds 6-7)
        n_est = int(np.mean(iters29))
        print(f'  refit n_estimators basis (seed-42 fold mean best_iteration): {n_est}')
        train_c, (test_c,) = add_counts(features, [test_features])
        test_probabilities = np.zeros(len(test))
        for seed in [42, 43, 44]:
            m = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63, n_estimators=n_est,
                                   n_jobs=-1, random_state=seed, verbose=-1, **STOCH)
            m.fit(to_lgbm(train_c), target)
            test_probabilities += m.predict_proba(to_lgbm(test_c))[:, 1] / 3
    else:  # '30'
        features_m2 = features.copy()
        features_m2['delay_was_missing'] = features_m2[DELAY_COL].isna().astype(int)
        test_m = test_features.copy()
        test_m['delay_was_missing'] = test_m[DELAY_COL].isna().astype(int)
        final = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63,
                                   n_estimators=int(np.mean(iters30)), n_jobs=-1,
                                   random_state=RANDOM_STATE, verbose=-1)
        final.fit(to_lgbm(features_m2), target)
        test_probabilities = final.predict_proba(to_lgbm(test_m))[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round9.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round9_oof.npz',
         oof_inc=np.asarray(oof_inc), aucs_inc=np.asarray(aucs_inc), iters_inc=np.asarray(iters_inc),
         oof29=np.asarray(oof29), aucs29=np.asarray(aucs29), iters29=np.asarray(iters29),
         oof30=np.asarray(oof30), aucs30=np.asarray(aucs30), iters30=np.asarray(iters30),
         y=np.asarray(target).astype(int))
meta = {
    'round': 9,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold; value counts fold-train-only',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'incumbent': 'LightGBM lr=0.05, num_leaves=63, n_estimators=1000, early stop 50 (Exp 8 config, recomputed in-run)',
        'exp29': 'stochastic(bagging 0.9/freq1, feature_fraction 0.9) + 21 fold-train-only count features; seeds 42/43/44 averaged',
        'exp30': 'incumbent config + delay_was_missing indicator (NaN left native)',
    },
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round9_oof.json').write_text(_json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


## What happens next

- If Exp 29 beats the bar: the two leads stack - it becomes the preferred candidate and the
  follow-ups are counts on the config mix, or smoothed/regularized count variants.
- If Exp 29 ties Exp 27 but beats Exp 22: averaging adds nothing to counts (overlapping
  mechanisms) - the simpler counts-only model is the cleaner candidate story for Andrea.
- If Exp 30 wins: missingness is signal - extend deliberate handling to other columns and re-test
  the forest with the indicator.
- If both null: the descriptive picture stays "one averaging win + one feature lead, everything
  else flat"; the external original-survey dataset remains the untried lever with third-party
  evidence behind it, gated on Andrea's explicit OK, and heavier stacking is the on-data option
  with published precedent (third-party claims, not our results).
